In [ ]:
!pip install -U streamlit langchain langchain-cohere langchain-community langchain-text-splitters langchain-chroma pypdf chromadb cohere rank_bm25 pyngrok -q

#Streamlit App

In [ ]:
%%writefile app.py
import os
import time
import glob
import streamlit as st
import cohere

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_cohere import CohereEmbeddings, ChatCohere
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# =============================================================================
# PAGE CONFIGURATION & STYLING
# =============================================================================
st.set_page_config(
    page_title="Universal Multi-PDF RAG Hub",
    page_icon="📚",
    layout="wide"
)

# Sidebar - Assignment Navigation
st.sidebar.title("📌 Assignment Navigation")
app_mode = st.sidebar.radio(
    "Select Part:",
    [
        "Part 1: Single PDF Basic RAG",
        "Part 2: Dynamic Chunking & Comparison",
        "Part 3: Multi-Doc RAG with Reranking"
    ]
)

st.sidebar.divider()

# Sidebar - Cohere API Key Input
api_key = st.sidebar.text_input("🔑 Cohere API Key", type="password")

if not api_key:
    st.warning("⚠️ Please enter your Cohere API Key in the sidebar to proceed.")
    st.stop()

# Initialize Embeddings & LLM
embeddings = CohereEmbeddings(model="embed-v4.0", cohere_api_key=api_key)
llm = ChatCohere(model="command-a-03-2025", temperature=0, cohere_api_key=api_key)

st.sidebar.divider()

# Sidebar - Universal PDF Document Uploader
st.sidebar.subheader("📂 Upload Documents")
uploaded_files = st.sidebar.file_uploader(
    "Upload any PDF(s) to process:",
    type=["pdf"],
    accept_multiple_files=True,
    key="global_pdf_uploader"
)

# Save uploaded PDFs to disk
saved_pdf_paths = []
if uploaded_files:
    os.makedirs("/content/uploaded_pdfs", exist_ok=True)
    for uploaded_file in uploaded_files:
        save_path = os.path.join("/content/uploaded_pdfs", uploaded_file.name)
        with open(save_path, "wb") as f:
            f.write(uploaded_file.getbuffer())
        saved_pdf_paths.append(save_path)
    st.sidebar.success(f"✅ {len(saved_pdf_paths)} PDF(s) active on server!")

# Helper function for rate-limit safe batch indexing
def build_vectorstore_safely(chunks, collection_name):
    vectorstore = Chroma(collection_name=collection_name, embedding_function=embeddings)
    BATCH_SIZE, DELAY = 15, 4
    progress_bar = st.progress(0, text="Indexing chunks into ChromaDB...")

    for i in range(0, len(chunks), BATCH_SIZE):
        batch = chunks[i : i + BATCH_SIZE]
        vectorstore.add_documents(documents=batch)
        pct = min((i + BATCH_SIZE) / len(chunks), 1.0)
        progress_bar.progress(pct, text=f"Indexed {min(i + BATCH_SIZE, len(chunks))}/{len(chunks)} chunks...")
        if i + BATCH_SIZE < len(chunks):
            time.sleep(DELAY)

    progress_bar.empty()
    return vectorstore

# Prompt template
prompt_template = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Answer the user's question using ONLY the provided context below.

If the answer cannot be found in the context,
say "I don't know."

Context:
{context}

Question:
{question}
""")

# =============================================================================
# PART 1: SINGLE PDF BASIC RAG
# =============================================================================
if app_mode == "Part 1: Single PDF Basic RAG":
    st.header("📄 Part 1: Single Document RAG Pipeline")
    st.markdown("Select any uploaded PDF from the sidebar to perform basic vector semantic search.")

    if not saved_pdf_paths:
        st.info("👈 Please upload at least one PDF file using the sidebar uploader.")
    else:
        selected_file = st.selectbox("Select PDF to query:", saved_pdf_paths, format_func=os.path.basename)

        if st.button("🚀 Process & Index Selected Document", key="p1_build"):
            with st.spinner("Processing document..."):
                loader = PyPDFLoader(selected_file)
                docs = loader.load()
                for d in docs:
                    d.metadata["source_file"] = os.path.basename(selected_file)

                splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
                chunks = splitter.split_documents(docs)

                vstore = build_vectorstore_safely(chunks, f"p1_{os.path.basename(selected_file)}")
                st.session_state["p1_retriever"] = vstore.as_retriever(search_kwargs={"k": 4})
                st.session_state["p1_active_doc"] = os.path.basename(selected_file)
                st.success(f"✅ Indexed `{os.path.basename(selected_file)}` successfully!")

        if "p1_retriever" in st.session_state:
            st.subheader(f"💬 Ask Questions (`{st.session_state.get('p1_active_doc')}`)")
            q = st.text_input("Enter your question:", key="p1_q")
            if q:
                retrieved_docs = st.session_state["p1_retriever"].invoke(q)
                context_str = "\n\n".join(d.page_content for d in retrieved_docs)
                res = (prompt_template | llm).invoke({"context": context_str, "question": q})

                st.markdown(f"**🤖 Answer:**\n{res.content}")
                with st.expander("📄 View Retrieved Chunks"):
                    for d in retrieved_docs:
                        st.caption(f"Page {d.metadata.get('page')}:")
                        st.text(d.page_content[:300] + "...")

# =============================================================================
# PART 2: DYNAMIC CHUNKING & COMPARISON EXPERIMENT
# =============================================================================
elif app_mode == "Part 2: Dynamic Chunking & Comparison":
    st.header("📚 Part 2: Dynamic Chunking & Side-by-Side Comparison")
    st.markdown("Choose any PDF and dynamically adjust chunk sizes and overlaps to evaluate retrieval performance.")

    if not saved_pdf_paths:
        st.info("👈 Please upload at least one PDF file using the sidebar uploader.")
    else:
        selected_file = st.selectbox("Select PDF for chunking experiment:", saved_pdf_paths, format_func=os.path.basename, key="p2_file_select")

        st.subheader("⚙️ Chunking Parameters")
        enable_comparison = st.checkbox("🔄 Enable Side-by-Side Comparison Mode", value=True)

        col_cfg1, col_cfg2 = st.columns(2)

        with col_cfg1:
            st.markdown("### Config A")
            chunk_size_a = st.slider("Chunk Size A (chars)", min_value=200, max_value=2000, value=800, step=50)
            chunk_overlap_a = st.slider("Chunk Overlap A", min_value=0, max_value=300, value=100, step=10)

        if enable_comparison:
            with col_cfg2:
                st.markdown("### Config B")
                chunk_size_b = st.slider("Chunk Size B (chars)", min_value=200, max_value=2000, value=400, step=50)
                chunk_overlap_b = st.slider("Chunk Overlap B", min_value=0, max_value=300, value=50, step=10)

        if st.button("🚀 Index Document with Custom Parameters"):
            loader = PyPDFLoader(selected_file)
            docs = loader.load()

            st.info(f"Indexing Configuration A (Size: {chunk_size_a}, Overlap: {chunk_overlap_a})...")
            splitter_a = RecursiveCharacterTextSplitter(chunk_size=chunk_size_a, chunk_overlap=chunk_overlap_a)
            chunks_a = splitter_a.split_documents(docs)
            vstore_a = build_vectorstore_safely(chunks_a, f"p2_cfg_a_{chunk_size_a}")
            st.session_state["p2_r_a"] = vstore_a.as_retriever(search_kwargs={"k": 4})
            st.session_state["cfg_a_label"] = f"Size: {chunk_size_a} | Overlap: {chunk_overlap_a}"

            if enable_comparison:
                st.info(f"Indexing Configuration B (Size: {chunk_size_b}, Overlap: {chunk_overlap_b})...")
                splitter_b = RecursiveCharacterTextSplitter(chunk_size=chunk_size_b, chunk_overlap=chunk_overlap_b)
                chunks_b = splitter_b.split_documents(docs)
                vstore_b = build_vectorstore_safely(chunks_b, f"p2_cfg_b_{chunk_size_b}")
                st.session_state["p2_r_b"] = vstore_b.as_retriever(search_kwargs={"k": 4})
                st.session_state["cfg_b_label"] = f"Size: {chunk_size_b} | Overlap: {chunk_overlap_b}"
            else:
                if "p2_r_b" in st.session_state:
                    del st.session_state["p2_r_b"]

            st.success("🎉 Vectorstore(s) indexed successfully!")

        if "p2_r_a" in st.session_state:
            st.divider()
            user_q = st.text_input("Enter question for comparison:", key="p2_q")

            if user_q:
                if enable_comparison and "p2_r_b" in st.session_state:
                    col_out_a, col_out_b = st.columns(2)

                    with col_out_a:
                        st.subheader(f"🅰️ Config A ({st.session_state.get('cfg_a_label')})")
                        docs_a = st.session_state["p2_r_a"].invoke(user_q)
                        context_a = "\n\n".join(d.page_content for d in docs_a)
                        res_a = (prompt_template | llm).invoke({"context": context_a, "question": user_q})

                        st.markdown(f"**🤖 Answer:**\n{res_a.content}")
                        with st.expander("📄 View Retrieved Chunks"):
                            for i, d in enumerate(docs_a, 1):
                                st.caption(f"**Chunk {i} (Length: {len(d.page_content)} chars | Page {d.metadata.get('page')}):**")
                                st.text(d.page_content)

                    with col_out_b:
                        st.subheader(f"🅱️ Config B ({st.session_state.get('cfg_b_label')})")
                        docs_b = st.session_state["p2_r_b"].invoke(user_q)
                        context_b = "\n\n".join(d.page_content for d in docs_b)
                        res_b = (prompt_template | llm).invoke({"context": context_b, "question": user_q})

                        st.markdown(f"**🤖 Answer:**\n{res_b.content}")
                        with st.expander("📄 View Retrieved Chunks"):
                            for i, d in enumerate(docs_b, 1):
                                st.caption(f"**Chunk {i} (Length: {len(d.page_content)} chars | Page {d.metadata.get('page')}):**")
                                st.text(d.page_content)
                else:
                    st.subheader(f"🅰️ Config A ({st.session_state.get('cfg_a_label')})")
                    docs_a = st.session_state["p2_r_a"].invoke(user_q)
                    context_a = "\n\n".join(d.page_content for d in docs_a)
                    res_a = (prompt_template | llm).invoke({"context": context_a, "question": user_q})
                    st.markdown(f"**🤖 Answer:**\n{res_a.content}")

# =============================================================================
# PART 3: MULTI-DOC RAG WITH RERANKING
# =============================================================================
elif app_mode == "Part 3: Multi-Doc RAG with Reranking":
    st.header("🤖 Part 3: Multi-Document RAG (All Uploaded PDFs)")
    st.markdown("Features **Hybrid BM25 + Vector Search**, **Cohere Rerank v3.5**, and explicit **Source Citation** across all uploaded documents.")

    if not saved_pdf_paths:
        st.info("👈 Please upload one or more PDFs in the sidebar to start multi-document indexing.")
    else:
        st.write(f"📁 **Detected PDFs to index ({len(saved_pdf_paths)}):**")
        for path in saved_pdf_paths:
            st.caption(f"- `{os.path.basename(path)}`")

        if st.button("🚀 Build Multi-Doc Vectorstore & Hybrid Index", key="p3_build"):
            all_chunks = []
            for path in saved_pdf_paths:
                fname = os.path.basename(path)
                loader = PyPDFLoader(path)
                docs = loader.load()
                for d in docs:
                    d.metadata["source_file"] = fname
                splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
                all_chunks.extend(splitter.split_documents(docs))

            st.session_state["p3_vstore"] = build_vectorstore_safely(all_chunks, "p3_multi_doc_all")
            st.session_state["p3_bm25"] = BM25Retriever.from_documents(all_chunks)
            st.session_state["p3_bm25"].k = 6
            st.success(f"Indexed {len(all_chunks)} chunks across {len(saved_pdf_paths)} document(s)!")

        if "p3_vstore" in st.session_state:
            def hybrid_rerank(query):
                sem_docs = st.session_state["p3_vstore"].as_retriever(search_kwargs={"k": 6}).invoke(query)
                bm25_docs = st.session_state["p3_bm25"].invoke(query)

                combined = {}
                for d in sem_docs + bm25_docs:
                    uid = f"{d.metadata.get('source_file')}_p{d.metadata.get('page')}_{hash(d.page_content)}"
                    if uid not in combined:
                        combined[uid] = d
                candidates = list(combined.values())[:6]

                client = cohere.Client(api_key=api_key)
                res = client.rerank(
                    model="rerank-v3.5",
                    query=query,
                    documents=[d.page_content for d in candidates],
                    top_n=3
                )
                return [candidates[r.index] for r in res.results]

            # Chat Interface
            if "p3_messages" not in st.session_state:
                st.session_state.p3_messages = []

            for msg in st.session_state.p3_messages:
                with st.chat_message(msg["role"]):
                    st.markdown(msg["content"])

            if user_query := st.chat_input("Ask a question across all uploaded documents..."):
                st.session_state.p3_messages.append({"role": "user", "content": user_query})
                with st.chat_message("user"):
                    st.markdown(user_query)

                retrieved = hybrid_rerank(user_query)
                context_str = "\n\n".join([f"[{d.metadata.get('source_file')} | Page {d.metadata.get('page')}]\n{d.page_content}" for d in retrieved])

                ans = (prompt_template | llm).invoke({"context": context_str, "question": user_query})

                with st.chat_message("assistant"):
                    st.markdown(ans.content)
                    with st.expander("📚 View Cited Source Documents"):
                        for d in retrieved:
                            st.write(f"📄 **{d.metadata.get('source_file')}** (Page {d.metadata.get('page')})")
                            st.caption(d.page_content[:200] + "...")

                st.session_state.p3_messages.append({"role": "assistant", "content": ans.content})

#Launch App

In [ ]:
import subprocess
import time

# 1. Start Streamlit server with CORS and XSRF disabled for tunnels
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.address", "0.0.0.0",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
])

# Give Streamlit 3 seconds to spin up completely
time.sleep(3)

# 2. Connect via localhost.run
!ssh -o StrictHostKeyChecking=no -R 80:localhost:8501 nokey@localhost.run